# Stage 3. Paired tests and robustness diagnosticsEvery method predicts the same withheld fires, so all comparisons are paired.Absolute errors go to Wilcoxon, correlations to Steiger, and each family ofp-values is Holm-adjusted together.

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / "src"))from fireanalogs.config import Configcfg = Config.load(Path.cwd().parent / "config" / "default.yaml")DATA = Path.cwd().parent / cfg["paths"]["data_root"]RESULTS = Path.cwd().parent / cfg["paths"]["results_root"]RESULTS.mkdir(parents=True, exist_ok=True)print("data:", DATA)

In [ ]:
import pandas as pdfrom fireanalogs import similarity, statistics, robustnessfrom fireanalogs.library import embedding_matrix, fuel_slope_matrixlibrary = pd.read_csv(RESULTS / "fire_library_merged.csv", parse_dates=["event_date"])loo = pd.read_csv(RESULTS / "loo_results.csv", parse_dates=["event_date"])cosine = similarity.cosine_similarity_matrix(embedding_matrix(library, cfg.n_emb_dims))

## Embedding retrieval against each baseline

In [ ]:
comparison = statistics.compare_methods(loo)comparison.to_csv(RESULTS / "detailed_comparison_steiger_wilcoxon_holm.csv", index=False)comparison

## Sensitivity to analog neighbourhood sizeThe published K is the smallest value at which error has substantiallystabilised, which shows the result is not driven by a narrow neighbourhood orby one highly similar fire.

In [ ]:
k_table = robustness.k_sensitivity(library, cosine, cfg.k_list, prior_only=cfg.prior_only)k_table.to_csv(RESULTS / "k_sensitivity.csv", index=False)k_table.round(4)

## Transfer function: full fuel by slope against slope onlyRetrieval is held fixed at embedding similarity, so this isolates the transferfunction rather than the retrieval space.

In [ ]:
ablation = robustness.transfer_function_ablation(    library, cosine, fuel_slope_matrix(library),    k=cfg.k_default, prior_only=cfg.prior_only,)ablation.to_csv(RESULTS / "fuelslope_vs_slope_only.csv", index=False)robustness.summarise_transfer_ablation(ablation)

## Do ensembles help?Mixed results here support the reading that the learned representation carriesinformation the baselines do not, and that averaging it with weaker criteriadilutes rather than improves it.

In [ ]:
loo_ens = robustness.add_ensembles(loo)ensemble_table = robustness.compare_ensembles(loo_ens)ensemble_table.to_csv(RESULTS / "ensemble_comparison_holm.csv", index=False)ensemble_table